# Write and use a Method

Write and run a simple Method that queries by group. Then edit the code, combine multiple queries and build a DoWhy analysis.

The exercises use prepared example data. Finally, apply the same Methods to real Cube data.

## 1. Setup

If you ran this notebook before, restart the kernel and run it from the first cell. An open kernel may still hold an older version of the practice data.

Import the authoring API and practice data. `example_data` contains the materials for this notebook.

In [ ]:
from pathlib import Path
import sys

repository = next(
    path
    for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "src/decision_layer").is_dir()
)
if str(repository) not in sys.path:
    sys.path.insert(0, str(repository))

import pandas as pd
from IPython.display import display

from decision_layer.dev import MethodSession
from decision_layer.methods import (
    Dimension, Entity, Estimate, Filter, Measure, Refused, Scope, Table, method,
)
import examples.methods.patterns.teaching_data as example_data


def show_table(data):
    if isinstance(data, dict):
        data = [{"field": key, "value": value} for key, value in data.items()]
    display(pd.DataFrame(data))

import importlib
import examples.methods.patterns.scenarios as practice_scenarios

importlib.reload(practice_scenarios)
importlib.reload(example_data)


## 2. Create a Method that queries by group

Create a Method that queries several measures by one or more dimensions. Return the top `limit` groups, sorted by the first measure.

In [ ]:
@method("workshop.by_group", version="1.0.0", interpretation="descriptive")
async def my_by_group(ctx, *, selected_measures: list[Measure],
                      group_by: list[Dimension], limit: int = 3) -> Table:
    data = await ctx.aggregate(
        measures={f"value_{i}": ref for i, ref in enumerate(selected_measures, 1)},
        dimensions={f"group_{i}": ref for i, ref in enumerate(group_by, 1)},
        order=[(selected_measures[0], "desc")],
        limit=limit,
    )
    return Table.from_dataset(data, require_non_empty=True)

`selected_measures` and `group_by` are lists. Use a one-item list even when you need just one measure or dimension. `ctx.aggregate(...)` queries every measure for each combination of dimensions.

- `measures`: name the result columns `value_1`, `value_2`, and so on.
- `dimensions`: name the grouping columns `group_1`, `group_2`, and so on.
- `order`, `limit`: return the top `limit` groups sorted by the first measure.

### From an AI request to Method execution

Example: “Show the top 3 departments by retailer receipts for store 364 in February 2001.”

When you ask the AI, it finds the measure and grouping dimension in the semantic layer and fills in the inputs. In this notebook, you enter the same values in the execution code below.

- `selected_measures`: put the retailer receipts measure `cube://journey/transaction/receipts` in a list.
- `group_by`: put the department dimension `cube://journey/product/department` in a list.
- `limit`: 3
- `scope`: February 2001, transaction date, store 364 filter

`cube://journey/transaction/receipts` means the `receipts` measure in the `transaction` model of the `journey` connection. The practice code uses `cube://practice/observations/value` from prepared example data.

### Try the Method in the notebook

Test the `my_by_group` you just wrote with a prepared example response.

- `example_session("by_group")`: create a practice session that uses the example response.
- `register(my_by_group)`: register the Method in this session.
- `run(...)`: provide the measure, group, and period to run it.

In the product service, the Method is registered on the server ahead of time. When an AI makes an MCP request, it sends the Method name and inputs to the server. The server creates `ctx` and runs it.

In [ ]:
group_session = await example_data.example_session("by_group")
group_session.register(my_by_group)

trial = await group_session.run(
    my_by_group,
    selected_measures=["cube://practice/observations/value"],
    group_by=["cube://practice/observations/group"],
    scope=Scope(
        ("2026-02-01", "2026-02-28"),
        "cube://practice/observations/date",
    ),
    limit=3,
)

assert trial.result.status == "success"
assert [row["value_1"] for row in trial.result.primary.data] == [30, 20, 10]
assert len(trial.attempts) == 1
show_table(trial.result.primary.data)

group_session.close()

`group_session.run(...)` creates `ctx` with the period and filters from `scope`, then calls `my_by_group`. The Method requests a grouped query through `ctx.aggregate(...)`. The practice session checks the request and returns the prepared values `30, 20, 10`.

### Inspect the query record

Check the requested period, filters and execution status.

In [ ]:
show_table([
    {
        "query": index,
        "label": attempt.label,
        "status": attempt.status,
        "period": attempt.spec.time.date_range,
        "filters": [item.model_dump() for item in attempt.spec.filters],
    }
    for index, attempt in enumerate(trial.attempts, start=1)
])

### Use multiple measures and dimensions

Pass two measures and two dimensions to the same Method. It queries both measures for each `group_1` and `group_2` combination, sorted by the first measure.

In [ ]:
multi_session = await example_data.example_session("multi_group")
multi_session.register(my_by_group)

multi_trial = await multi_session.run(
    my_by_group,
    selected_measures=[
        "cube://practice/observations/value",
        "cube://practice/observations/before_value",
    ],
    group_by=[
        "cube://practice/observations/group",
        "cube://practice/observations/subject",
    ],
    scope=Scope(("2026-02-01", "2026-02-28"), "cube://practice/observations/date"),
    limit=3,
)

assert multi_trial.result.status == "success"
assert multi_trial.result.primary.data[0] == {
    "value_1": 30, "value_2": 25, "group_1": "A", "group_2": "X",
}
show_table(multi_trial.result.primary.data)
multi_session.close()

## 3. Add a rank

Add `rank` to the query result. The change starts after `data.records()`. The edited function uses the same Method name and a new version, `1.1.0`.

In [ ]:
@method("workshop.by_group", version="1.1.0", interpretation="descriptive")
async def my_ranked_by_group(ctx, *, selected_measures: list[Measure],
                             group_by: list[Dimension], limit: int = 3) -> Table:
    data = await ctx.aggregate(
        measures={f"value_{i}": ref for i, ref in enumerate(selected_measures, 1)},
        dimensions={f"group_{i}": ref for i, ref in enumerate(group_by, 1)},
        order=[(selected_measures[0], "desc")],
        limit=limit,
    )
    rows = data.records()
    for rank, row in enumerate(rows, start=1):
        row["rank"] = rank
    return Table(rows)

### Run the edited function

Register only the edited Method in a new practice session and run it. The result adds ranks `1, 2, 3`.

In [ ]:
ranked_session = await example_data.example_session("by_group")
ranked_session.register(my_ranked_by_group)

edited_trial = await ranked_session.run(
    my_ranked_by_group,
    selected_measures=["cube://practice/observations/value"],
    group_by=["cube://practice/observations/group"],
    scope=Scope(
        ("2026-02-01", "2026-02-28"),
        "cube://practice/observations/date",
    ),
    limit=3,
)

assert edited_trial.result.status == "success"
assert [row["rank"] for row in edited_trial.result.primary.data] == [1, 2, 3]
assert [row["value_1"] for row in edited_trial.result.primary.data] == [30, 20, 10]
show_table(edited_trial.result.primary.data)

ranked_session.close()

## 4. Combine queries: Peer Comparison

Run three queries in one function and combine their values in one table.

| Query | Population |
| --- | --- |
| subject | A in group X |
| peers | Group X excluding A |
| overall | Everyone excluding A |

`ctx.scope.filters` contains conditions applied to every query. The function below checks them so the comparison population is not also restricted to subject A.

In [ ]:
@method(
    "workshop.peer",
    version="1.0.0",
    interpretation="descriptive",
)
async def my_peer(
    ctx,
    *,
    measure: Measure,
    subject: Dimension,
    category: Dimension,
    subject_value: str,
    category_value: str,
) -> Table:
    if any(item.member == subject for item in ctx.scope.filters):
        raise Refused("Keep the subject out of shared filters")

    same_category = Filter(
        member=category,
        operator="equals",
        values=[category_value],
    )
    selected = Filter(member=subject, operator="equals", values=[subject_value])
    excluded = Filter(member=subject, operator="notEquals", values=[subject_value])

    subject_data = await ctx.aggregate(
        measures={"value": measure},
        filters=[same_category, selected],
        label="subject",
    )
    peer_data = await ctx.aggregate(
        measures={"value": measure},
        filters=[same_category, excluded],
        label="peers",
    )
    overall_data = await ctx.aggregate(
        measures={"value": measure},
        filters=[excluded],
        label="overall",
    )

    subject_result = subject_data.scalar("value")
    peer_result = peer_data.scalar("value")
    overall_result = overall_data.scalar("value")

    return Table(
        [
            {
                "comparison": "peers",
                "subject": subject_result,
                "benchmark": peer_result,
                "difference": subject_result - peer_result,
            },
            {
                "comparison": "overall",
                "subject": subject_result,
                "benchmark": overall_result,
                "difference": subject_result - overall_result,
            },
        ],
        warnings=["Descriptive comparison; no causal claim or significance test."],
    )

### Register and run

Select subject A and group X. The prepared responses are `12, 8, 6`, giving differences of `4, 6`.

In [ ]:
peer_session = await example_data.example_session("peer")
peer_session.register(my_peer)

peer_trial = await peer_session.run(
    my_peer,
    measure="cube://practice/observations/rate",
    subject="cube://practice/observations/subject",
    category="cube://practice/observations/group",
    scope=Scope(
        ("2026-02-01", "2026-02-28"),
        "cube://practice/observations/date",
    ),
    subject_value="A",
    category_value="X",
)

assert peer_trial.result.status == "success"
assert [row["difference"] for row in peer_trial.result.primary.data] == [4, 6]
assert len(peer_trial.attempts) == 3
show_table(peer_trial.result.primary.data)

peer_session.close()

### Inspect the three queries

Compare their labels and filters. As an editing exercise, add `ratio` to the result.

In [ ]:
show_table([
    {
        "query": index,
        "label": attempt.label,
        "status": attempt.status,
        "filters": [item.model_dump() for item in attempt.spec.filters],
    }
    for index, attempt in enumerate(peer_trial.attempts, start=1)
])

## 5. DoWhy regression adjustment

Use DoWhy's `backdoor.linear_regression` to adjust a binary treatment's `0→1` contrast.

| Input | Meaning |
| --- | --- |
| outcome | Outcome measure |
| treatment | Treatment coded 0 or 1 |
| confounders | Pretreatment adjustment variables |
| unit | Observation unit |

Data checks do not verify identification assumptions. This example reports an associational result without a confidence interval.

### Write the calculation function

Check data, preprocess it, inspect support and estimate with DoWhy. The same code is available in [binary_regression.py](binary_regression.py).

In [ ]:
def estimate_binary_regression(frame, confounder_columns):
    """Estimate the 0-to-1 contrast under an explicitly assumed backdoor graph."""
    import networkx as nx
    import numpy as np
    import pandas as pd
    from dowhy import CausalModel
    from sklearn.linear_model import LogisticRegression

    # 1. Check the observation units, missing values and treatment groups.
    if (
        frame.empty
        or frame["unit"].isna().any()
        or frame["unit"].duplicated().any()
    ):
        raise Refused("Unique non-null analysis units are required")

    required_columns = ["outcome", "treatment", *confounder_columns]
    if not confounder_columns or frame[required_columns].isna().any().any():
        raise Refused(
            "Observed adjustment variables and a missing-data policy are required"
        )

    if (
        set(frame["treatment"].unique()) != {0, 1}
        or frame.groupby("treatment").size().min() < 30
    ):
        raise Refused(
            "This example requires binary treatment and at least 30 units per arm"
        )

    # 2. Bound encoding before allocating the matrix.
    encoded_width = sum(
        1 if pd.api.types.is_numeric_dtype(frame[column]) else max(frame[column].nunique() - 1, 0)
        for column in confounder_columns
    )
    if encoded_width == 0 or encoded_width > 100 or encoded_width * len(frame) > 2_000_000:
        raise Refused("Adjustment encoding is empty or exceeds the matrix bound")
    encoded = pd.get_dummies(
        frame[confounder_columns],
        drop_first=True,
        dtype=float,
    )
    if (
        not len(encoded.columns)
        or len(encoded.columns) > 100
        or encoded.size > 2_000_000
    ):
        raise Refused("Adjustment encoding is empty or exceeds the matrix bound")

    encoded.columns = [f"adjustment_{i}" for i in range(len(encoded.columns))]

    if (
        not np.isfinite(frame["outcome"].to_numpy(dtype=float)).all()
        or not np.isfinite(encoded.to_numpy()).all()
    ):
        raise Refused("Finite outcomes and adjustment variables are required")

    observations = pd.concat(
        [
            frame[["outcome", "treatment"]].reset_index(drop=True),
            encoded.reset_index(drop=True),
        ],
        axis=1,
    )

    predictor_columns = ["treatment", *encoded.columns]
    predictors = observations[predictor_columns].to_numpy(dtype=float)
    matrix = np.column_stack([np.ones(len(frame)), predictors])

    if (
        len(frame) <= matrix.shape[1]
        or np.linalg.matrix_rank(matrix) < matrix.shape[1]
    ):
        raise Refused(
            "The regression needs full-rank predictors and residual degrees of freedom"
        )

    # 3. Diagnose observed treatment support. This does not prove positivity.
    propensity = LogisticRegression(
        max_iter=1000,
        random_state=7,
    ).fit(encoded, frame["treatment"])
    probability = propensity.predict_proba(encoded)[:, 1]

    if np.any(probability < 0.05) or np.any(probability > 0.95):
        raise Refused(
            "Observed support fails this example's diagnostic; not a positivity proof"
        )

    # 4. Declare the assumed DAG and identify its backdoor estimand.
    edges = [("treatment", "outcome")]
    edges += [
        (column, target)
        for column in encoded.columns
        for target in ("treatment", "outcome")
    ]
    model = CausalModel(
        data=observations,
        treatment="treatment",
        outcome="outcome",
        graph=nx.DiGraph(edges),
    )
    estimand = model.identify_effect(proceed_when_unidentifiable=False)

    if not (estimand.estimands.get("backdoor") or {}).get("estimand"):
        raise Refused("No backdoor estimand is identified under the assumed graph")

    # 5. Fit the explicit binary regression contrast and a placebo diagnostic.
    estimate = model.estimate_effect(
        estimand,
        method_name="backdoor.linear_regression",
        control_value=0,
        treatment_value=1,
        target_units="ate",
        confidence_intervals=False,
    )
    placebo = model.refute_estimate(
        estimand,
        estimate,
        method_name="placebo_treatment_refuter",
        placebo_type="permute",
        num_simulations=20,
        random_state=7,
    )

    # 6. Return the estimate with its analysis settings and interpretation limits.
    return {
        "value": float(estimate.value),
        "estimator": "backdoor.linear_regression",
        "contrast": [0, 1],
        "target": "ate",
        "n_units": len(frame),
        "graph_edges": edges,
        "estimand": str(estimand),
        "placebo_difference": float(placebo.new_effect),
        "seed": 7,
        "support": {
            "min": float(probability.min()),
            "max": float(probability.max()),
        },
        "causal_claim": "not_established",
        "uncertainty": "not_reported",
    }

### Connect it as a Method

Use `ctx.entities(...)` to retrieve data per `unit`, then `ctx.compute(...)` to run the calculation function above.

In [ ]:
@method(
    "workshop.binary_regression",
    version="1.0.0",
    interpretation="associational",
    execution="dataframe",
    requires_packages={
        "dowhy": "==0.13",
        "numpy": ">=1.26,<2",
        "pandas": ">=2.2,<3",
        "scikit-learn": ">=1.2,<1.7",
    },
)
async def my_binary_regression(
    ctx,
    *,
    outcome: Measure,
    treatment: Dimension,
    confounders: list[Dimension | Measure],
    unit: Entity,
) -> Estimate:
    # Extract governed entity data, then run ordinary Python calculation code.
    observations = await ctx.entities(
        entity=unit,
        measures={"outcome": outcome},
        dimensions={
            "treatment": treatment,
            "confounders": confounders,
        },
    )
    payload = await ctx.compute(
        estimate_binary_regression,
        observations.frame,
        observations.columns("confounders"),
    )

    return Estimate(
        payload["value"],
        details={
            **payload,
            "assumptions_status": "not_verified",
        },
        warnings=[
            "Unverified assumptions and placebo diagnostics do not establish "
            "causality; no interval is reported."
        ],
    )

### Register and run

Connect the outcome, treatment, adjustment variables and observation unit. The example data follows `Y = 2*T + 3*W + noise`, with an expected value near `2`.

In [ ]:
regression_session = await example_data.example_session("dowhy")
regression_session.register(my_binary_regression)

dowhy_trial = await regression_session.run(
    my_binary_regression,
    outcome="cube://practice/observations/value",
    treatment="cube://practice/observations/treated",
    confounders=["cube://practice/observations/baseline"],
    unit="cube://practice/observations/id",
    scope=Scope(
        ("2026-02-01", "2026-02-28"),
        "cube://practice/observations/date",
    ),
)

assert dowhy_trial.result.status == "success"
assert abs(dowhy_trial.result.primary.data["value"] - 2) < 0.15
show_table(dowhy_trial.result.primary.data)

regression_session.close()


### Use it in ordinary Python

Pass a DataFrame directly to the same calculation function and compare its value with the Method result.

In [ ]:
frame = example_data.binary_regression_frame()
standalone_result = estimate_binary_regression(frame, ["baseline"])

assert abs(standalone_result["value"] - 2) < 0.15
assert standalone_result["value"] == dowhy_trial.result.primary.data["value"]
show_table(standalone_result)


### Inspect a refusal

Use practice data containing a duplicate observation unit. The Method refuses estimation and returns a reason.

In [ ]:
refusal_session = await example_data.duplicate_unit_session()
refusal_session.register(my_binary_regression)

refused_trial = await refusal_session.run(
    my_binary_regression,
    outcome="cube://practice/observations/value",
    treatment="cube://practice/observations/treated",
    confounders=["cube://practice/observations/baseline"],
    unit="cube://practice/observations/id",
    scope=Scope(
        ("2026-02-01", "2026-02-28"),
        "cube://practice/observations/date",
    ),
)

assert refused_trial.result.status == "refused"
show_table([
    {"status": refused_trial.result.status, "reason": warning}
    for warning in refused_trial.result.warnings
])
refusal_session.close()


## 6. Use the Methods with Cube

Set `RUN_LIVE` to `True` in the connection cell. The default URL and secret are for the local sample. Find the model and member paths in the real catalog, then run the functions defined above.

In [ ]:
import os
from decision_layer.semantic.credentials import ServiceCredentials

RUN_LIVE = False
live = None

if RUN_LIVE:
    live = await MethodSession.cube(
        os.environ.get("DL_JOURNEY_CUBE_URL", "http://127.0.0.1:4000/cubejs-api/v1"),
        ServiceCredentials(
            os.environ.get(
                "DL_JOURNEY_CUBE_SECRET",
                "local-example-secret-change-me-0123456789",
            ),
            (),
        ),
        instance="journey",
    )

show_table([{
    "status": "connected" if live is not None else "skipped",
    "reason": "Cube connected" if live is not None else "Set RUN_LIVE = True in this connection cell",
}])

### Find the objects

Find the receipts measure and department dimension in the connected Cube. The catalog lists available objects; inside a Method, you can also access it through `ctx.catalog`.

In [ ]:
if live is not None:
    objects = [
        *live.catalog.search(kind="measure", text="receipts"),
        *live.catalog.search(kind="dimension", text="department"),
    ]
    show_table([
        {"title": obj.title, "kind": obj.kind, "path": obj.ref}
        for obj in objects
    ])
else:
    show_table([{"status": "skipped", "reason": "Enable the Cube connection above"}])

### Query by group

Query retailer receipts by department. Try changing `limit` or the period.

In [ ]:
if live is not None:
    live.register(my_by_group, replace=True)
    live_group_trial = await live.run(
        my_by_group,
        selected_measures=["cube://journey/transaction/receipts"],
        group_by=["cube://journey/product/department"],
        scope=Scope(
            ("2001-01-01", "2001-06-30"),
            "cube://journey/transaction/analysis_date",
        ),
        limit=3,
    )
    show_table(live_group_trial.result.primary.data)
else:
    show_table([{"status": "skipped", "reason": "Enable the Cube connection above"}])

### Compare populations

Compare store 364 in GROCERY with its peers and the overall population. Try other stores or departments present in the data.

In [ ]:
if live is not None:
    live.register(my_peer, replace=True)
    live_peer_trial = await live.run(
        my_peer,
        measure="cube://journey/transaction/coupon_line_rate",
        subject="cube://journey/transaction/store",
        category="cube://journey/product/department",
        scope=Scope(
            ("2001-01-01", "2001-06-30"),
            "cube://journey/transaction/analysis_date",
        ),
        subject_value="364",
        category_value="GROCERY",
    )
    show_table(live_peer_trial.result.primary.data)
    show_table([
        {"query": index, "label": attempt.label, "status": attempt.status}
        for index, attempt in enumerate(live_peer_trial.attempts, start=1)
    ])
else:
    show_table([{"status": "skipped", "reason": "Enable the Cube connection above"}])

### Close the connection

Close the local session. To serve a Method, save it in a reviewed Python file and register it on the server.

In [ ]:
if live is not None:
    live.close()
    live = None

show_table([{"status": "closed"}])